# Lesson 12 — SPI 事务必须由片选划定边界 — 参考实现

先完成主 Notebook 的练习再阅读。此 Notebook 只展示可通过 exercise 契约的一种实现。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


运行代码格，使用同一个可交互面板验证。然后回到主 Notebook，对照自己的状态和边界处理。


In [ ]:
%%academy_lab 12-spi-display solution exercise pass
#pragma once
#include "academy/device.hpp"
#include <string>
namespace academy {
    class Controller {
    public:
        explicit Controller(Device&);
        void tick();

    private:
        Device& board_;
        bool selected_ = false;
        std::string frame_;
    };
}

namespace academy {
    Controller::Controller(Device& b) : board_(b) {
    }
    void Controller::tick() {
        const std::string& x = board_.input.text;
        board_.output.error = false;
        if (board_.input.fault) {
            selected_ = false;
            frame_.clear();
            board_.output.error = true;
            board_.output.state = "spi error";
            return;
        }
        if (x == "CS:LOW") {
            selected_ = true;
            frame_.clear();
            board_.output.state = "selected";
        } else if (x == "CS:HIGH") {
            if (!selected_) {
                board_.output.error = true;
            } else {
                board_.output.display = frame_;
                board_.output.state = "spi ready";
            }
            selected_ = false;
        } else if (x.rfind("WRITE:", 0) == 0) {
            if (!selected_)
                board_.output.error = true;
            else
                frame_ += x.substr(6);
        } else if (!x.empty())
            board_.output.error = true;
    }
}
